# Battle Lab · Fase 2.1 — VGC-Bench bajo el microscopio

Esta es la libreta canónica del Battle Lab. Prepara un servidor privado de Pokémon Showdown, valida un corpus de equipos Champions M-C reales y mide el checkpoint público final de **VGC-Bench contra Random, Max Base Power y Simple Heuristics**. El self-play continúa disponible como modo alternativo.

Por defecto rota 12 equipos públicos de torneo/ladder documentados por VGCPastes sin repetir pareja en los primeros 66 combates. También añade cualquier equipo `.txt` exportado desde nuestro Team Builder y guardado en `Pokemon VGC/BattleLab/teams`. Todos pasan por el validador de Showdown antes de jugar.

**GPU recomendada, no obligatoria.** El benchmark ejecuta 500 combates por rival (1,500 en total), espeja Teams y lados, muestra progreso/ETA y calcula un Elo interno con intervalo de confianza. La sección de auditoría cruza derrotas con sus replays y combates espejo, y genera HTML, JSON, CSV y una muestra de replays prioritarios. Ese Elo sirve para comparar ejecuciones del laboratorio; no es el Elo oficial de Showdown.

> Nota honesta: el checkpoint fue entrenado con equipos M-A/M-B. Aquí lo evaluamos en M-C sin reentrenarlo, así que será mucho más serio que los bots del smoke test, pero no asumimos un Elo competitivo hasta medirlo.


## 1. Configuración

`RUN_EVALUATION = False` reutiliza el último benchmark guardado y evita repetir las 1,500 partidas; actívalo para una ejecución nueva. `RUN_AUDIT` genera la autopsia del equipo y baseline seleccionados. `RUN_MODE = "self-play"` usa `BATTLES`. `DEVICE = "auto"` exprime la GPU si está disponible. `DRIVE_TEAMS` recibe los `.txt` descargados desde el Team Builder.


In [ ]:
PKMN_REPOSITORY = "https://github.com/Iesyo/pkmn.git"  # @param {type:"string"}
PKMN_REF = "main"  # @param {type:"string"}
RUN_EVALUATION = False  # @param {type:"boolean"}
RUN_MODE = "benchmark"  # @param ["benchmark", "self-play"]
BENCHMARK_BATTLES_PER_BASELINE = 500  # @param {type:"integer"}
BATTLES = 20  # @param {type:"integer"}
RUN_AUDIT = True  # @param {type:"boolean"}
AUDIT_TEAM_ID = "MC182"  # @param {type:"string"}
AUDIT_BASELINE = "simple-heuristics"  # @param ["simple-heuristics", "max-base-power", "random"]
AUDIT_SAMPLE_SIZE = 20  # @param {type:"integer"}
DEVICE = "auto"  # @param ["auto", "cuda", "cpu"]
SEED = 260913  # @param {type:"integer"}
SAVE_TO_DRIVE = True  # @param {type:"boolean"}
DRIVE_RESULTS = "Pokemon VGC/BattleLab/results/phase-2"  # @param {type:"string"}
DRIVE_TEAMS = "Pokemon VGC/BattleLab/teams"  # @param {type:"string"}
RUNTIME_ROOT = "/content/battle-lab-runtime"
PKMN_ROOT = "/content/pkmn"
NODE_VERSION = "24.21.0"

if RUN_MODE not in {"benchmark", "self-play"}:
    raise ValueError("RUN_MODE debe ser benchmark o self-play")
if BENCHMARK_BATTLES_PER_BASELINE < 1:
    raise ValueError("BENCHMARK_BATTLES_PER_BASELINE debe ser mayor que cero")
if BATTLES < 1:
    raise ValueError("BATTLES debe ser mayor que cero")
if AUDIT_SAMPLE_SIZE < 1:
    raise ValueError("AUDIT_SAMPLE_SIZE debe ser mayor que cero")
if AUDIT_BASELINE not in {"simple-heuristics", "max-base-power", "random"}:
    raise ValueError("AUDIT_BASELINE no es válido")
if not AUDIT_TEAM_ID.strip() or not AUDIT_TEAM_ID.strip().replace("-", "").isalnum():
    raise ValueError("AUDIT_TEAM_ID contiene caracteres no válidos")


## 2. Preparar el código y las dependencias


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import time

pkmn_root = Path(PKMN_ROOT)
runtime_root = Path(RUNTIME_ROOT)
packages_root = runtime_root / "python-packages"
battle_lab_python = Path(sys.executable)
battle_lab_env = os.environ.copy()
previous_pythonpath = battle_lab_env.get("PYTHONPATH")
battle_lab_env["PYTHONPATH"] = str(packages_root) + (os.pathsep + previous_pythonpath if previous_pythonpath else "")
started = time.monotonic()
prep_total = 6

def prep_progress(completed, detail):
    elapsed = time.monotonic() - started
    eta = elapsed / completed * (prep_total - completed) if completed else None
    filled = round(24 * completed / prep_total)
    bar = "█" * filled + "░" * (24 - filled)
    eta_text = f"{eta:.0f}s" if eta is not None else "calculando"
    print(f"Preparación [{bar}] {completed}/{prep_total} · {elapsed:.0f}s · ETA {eta_text} · {detail}", flush=True)

def run_live(command, *, cwd=None, label, attempts=1, env=None):
    command = [str(part) for part in command]
    for attempt in range(1, attempts + 1):
        print(f"\n▶ {label} · intento {attempt}/{attempts}", flush=True)
        step_started = time.monotonic()
        process = subprocess.Popen(
            command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        lines = []
        assert process.stdout is not None
        for line in process.stdout:
            lines.append(line)
            print(line, end="", flush=True)
        return_code = process.wait()
        if return_code == 0:
            print(f"✅ {label} ({time.monotonic() - step_started:.1f}s)", flush=True)
            return
        if attempt < attempts:
            wait_seconds = 5 * attempt
            print(f"⚠️ {label} falló; reintento en {wait_seconds}s.", flush=True)
            time.sleep(wait_seconds)
            continue
        tail = "".join(lines[-60:]) or "<el comando no produjo salida>"
        raise RuntimeError(f"{label} falló tras {attempts} intento(s).\nÚltimas líneas:\n{tail}")

prep_progress(0, f"Python {sys.version.split()[0]}")
if not (pkmn_root / ".git").is_dir():
    run_live(
        ["git", "clone", "--filter=blob:none", "--no-checkout", PKMN_REPOSITORY, pkmn_root],
        label="Clonar pkmn",
    )
else:
    dirty = subprocess.run(
        ["git", "status", "--porcelain", "--untracked-files=no"],
        cwd=pkmn_root, text=True, capture_output=True, check=True,
    ).stdout.strip()
    if dirty:
        raise RuntimeError("El checkout temporal de pkmn contiene cambios; usa un runtime limpio.")

run_live(["git", "fetch", "--depth", "1", "origin", PKMN_REF], cwd=pkmn_root, label="Actualizar pkmn", attempts=3)
run_live(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=pkmn_root, label="Fijar revisión de pkmn")
prep_progress(1, "Código fijado")

node_result = subprocess.run(["node", "--version"], text=True, capture_output=True) if shutil.which("node") else None
node_major = int(node_result.stdout.strip().lstrip("v").split(".")[0]) if node_result and node_result.returncode == 0 else 0
if node_major == 0 or not shutil.which("npm"):
    if not shutil.which("apt-get"):
        raise RuntimeError("Este runtime no incluye Node.js/npm ni apt-get para preparar el instalador.")
    apt_env = os.environ.copy()
    apt_env["DEBIAN_FRONTEND"] = "noninteractive"
    run_live(["apt-get", "update"], label="Actualizar catálogo de Node.js", attempts=3, env=apt_env)
    run_live(["apt-get", "install", "-y", "nodejs", "npm"], label="Preparar instalador de Node.js", attempts=3, env=apt_env)
if node_major < 24:
    run_live(["npm", "install", "--global", "n@latest"], label="Instalar gestor de Node.js", attempts=3)
    run_live(["n", NODE_VERSION], label=f"Instalar Node.js {NODE_VERSION}", attempts=3)
run_live(["node", "--version"], label="Verificar Node.js")
run_live(["npm", "--version"], label="Verificar npm")
node_major = int(subprocess.check_output(["node", "--version"], text=True).strip().lstrip("v").split(".")[0])
if node_major < 24:
    raise RuntimeError(f"Battle Lab requiere Node.js >=24; el runtime todavía tiene {node_major}.")
prep_progress(2, f"Node.js y npm listos · {NODE_VERSION}")

packages_root.mkdir(parents=True, exist_ok=True)
prep_progress(3, "Directorio aislado listo")

run_live(
    [battle_lab_python, "-m", "pip", "install", "--target", packages_root, "--upgrade", "--retries", "10", "--timeout", "120", "-r", "battle_lab/requirements-phase1.txt"],
    cwd=pkmn_root, label="Instalar dependencias base del Battle Lab", attempts=3,
)
prep_progress(4, "Dependencias base instaladas")

run_live(
    [battle_lab_python, "-m", "pip", "install", "--target", packages_root, "--upgrade", "--no-deps", "--retries", "10", "--timeout", "120", "-r", "battle_lab/requirements-phase2.txt"],
    cwd=pkmn_root, label="Instalar inferencia VGC-Bench sin duplicar PyTorch/CUDA", attempts=3,
)
prep_progress(5, "VGC-Bench listo")

run_live(
    [battle_lab_python, "-c", "import poke_env, psutil, torch, stable_baselines3; print(f'VGC-Bench listo · torch {torch.__version__} · CUDA {torch.cuda.is_available()} · SB3 {stable_baselines3.__version__}')"],
    label="Verificar dependencias", env=battle_lab_env,
)
prep_progress(6, f"Todo listo en {time.monotonic() - started:.1f}s")


## 3. Conectar Google Drive

El JSON final y el ZIP de replays/logs se guardan en Drive. La carpeta de Teams sirve como puente entre sistemas: en la web usa **Exportar → Descargar .txt**, sube el archivo ahí y esta libreta lo incorporará automáticamente en la siguiente ejecución. Un archivo inválido se informa y se omite; no rompe el resto del corpus.


In [ ]:
from pathlib import Path

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    results_dir = Path("/content/drive/MyDrive") / DRIVE_RESULTS
    teams_dir = Path("/content/drive/MyDrive") / DRIVE_TEAMS
else:
    results_dir = Path(RUNTIME_ROOT) / "results"
    teams_dir = Path(RUNTIME_ROOT) / "teams"

results_dir.mkdir(parents=True, exist_ok=True)
teams_dir.mkdir(parents=True, exist_ok=True)
print(f"Resultados: {results_dir}")
print(f"Teams propios: {teams_dir} · {len(list(teams_dir.rglob('*.txt')))} archivo(s)")


## 4. Ejecutar benchmark o self-play en M-C


In [ ]:
command = [
    str(battle_lab_python),
    "battle_lab/vgc_bench_battle.py",
    "--runtime-root", RUNTIME_ROOT,
    "--results-dir", str(results_dir),
    "--extra-teams-dir", str(teams_dir),
    "--mode", RUN_MODE,
    "--benchmark-battles-per-baseline", str(BENCHMARK_BATTLES_PER_BASELINE),
    "--battles", str(BATTLES),
    "--device", DEVICE,
    "--seed", str(SEED),
]
if RUN_EVALUATION:
    run_live(command, cwd=pkmn_root, label="Evaluar a VGC-Bench sin piedad", env=battle_lab_env)
else:
    print("⏭️ Benchmark omitido: se reutilizará el resultado más reciente de Drive.")


## 5. Resumen verificable


In [ ]:
import json
from IPython.display import HTML, display

result_files = sorted(results_dir.glob("vgc-bench-*.json"), key=lambda path: path.stat().st_mtime)
if not result_files:
    raise RuntimeError("No se encontró el resultado de VGC-Bench")

latest_result = result_files[-1]
result = json.loads(latest_result.read_text(encoding="utf-8"))
battles = result["battles"]
showdown = result["showdown"]
vgc = result["vgcBench"]
policy = vgc["policy"]
teams = result["teams"]
wins = battles["wins"]
mode = result.get("mode", "self-play")
if mode == "benchmark":
    benchmark = result["benchmark"]
    baseline_rows = "".join(
        f"<tr><td style='padding:5px 12px 5px 0'>{report['label']}</td><td style='padding:5px 12px'>{report['wins']}-{report['losses']}-{report['ties']}</td><td style='padding:5px 12px'>{report['scorePercent']:.2f}%</td><td style='padding:5px 0 5px 12px'>{report['elo']['difference']:+.1f}</td></tr>"
        for report in benchmark["opponents"].values()
    )
    overall_benchmark = benchmark["overall"]
    title = "📊 Benchmark VGC-Bench completado"
    victories = f"VGC-Bench <b>{wins['vgcBench']}</b> · Baselines <b>{wins['baselines']}</b> · Empates <b>{wins['ties']}</b>"
    benchmark_panel = f"""
      <div style='margin-top:12px;padding:12px;border-radius:10px;background:#0f172a'>
        <div style='font-weight:800;color:#fbbf24'>Elo interno: {overall_benchmark['elo']['performanceRating']:.1f} · puntuación {overall_benchmark['scorePercent']:.2f}%</div>
        <table style='margin-top:8px;border-collapse:collapse'><thead><tr style='color:#94a3b8'><th style='text-align:left'>Rival</th><th>W-L-T</th><th>Puntuación</th><th>ΔElo</th></tr></thead><tbody>{baseline_rows}</tbody></table>
        <div style='margin-top:8px;color:#94a3b8'>Agenda espejada <code>{benchmark['schedule']['sha256'][:12]}</code> · cada rival anclado internamente en 1500 · no es Elo oficial de Showdown.</div>
      </div>
    """
else:
    title = "⚔️ Self-play VGC-Bench completado"
    victories = f"Alpha <b>{wins['alpha']}</b> · Beta <b>{wins['beta']}</b> · Empates <b>{wins['ties']}</b>"
    benchmark_panel = ""
display(HTML(f"""
<div style='padding:18px;border-radius:14px;background:#07111f;color:#e2e8f0;border:1px solid #164e63'>
  <div style='font-size:20px;font-weight:800;color:#67e8f9'>{title}</div>
  <div style='margin-top:10px'>Formato: <b>{showdown['format']}</b></div>
  <div>Showdown: <code>{showdown['commit'][:12]}</code></div>
  <div>VGC-Bench: <code>{vgc['commit'][:12]}</code> · checkpoint <code>{vgc['checkpoint']['sha256'][:12]}</code></div>
  <div>Política: <b>determinista</b> · Team Preview IA · dispositivo <b>{policy['device']}</b></div>
  <div>Corpus: <b>{teams['available']} equipos válidos</b> · {teams['rotation']['uniquePairings']} cruces únicos · {teams['byOrigin'].get('team-builder-drive', 0)} desde Team Builder/Drive</div>
  <div>Omitidos: <b>{len(teams['ignored'])}</b> · algoritmo <code>{teams['rotation']['mode']}</code></div>
  <div>Combates: <b>{battles['completed']}/{battles['requested']}</b></div>
  <div>Victorias: {victories}</div>
  <div>Rendimiento: <b>{battles['battlesPerMinute']} combates/min</b></div>
  {benchmark_panel}
  <div style='margin-top:10px;color:#94a3b8'>JSON: {latest_result}</div>
  <div style='color:#94a3b8'>Replays: {result['artifacts']['replaysZip']}</div>
</div>
"""))


## 6. Auditar derrotas y combates espejo

Esta sección no vuelve a ejecutar el benchmark. Lee el JSON y el ZIP más recientes, compara cada derrota del equipo elegido con su combate espejo y conserva una muestra diversa de casos para revisión humana. Los indicadores automáticos son evidencia observable, no una sentencia táctica.


In [ ]:
if RUN_AUDIT:
    if mode != "benchmark":
        raise RuntimeError("La auditoría requiere un resultado en modo benchmark.")
    replay_zip = Path(result["artifacts"]["replaysZip"])
    if not replay_zip.is_file():
        replay_zip = latest_result.with_name(f"{latest_result.stem}-replays.zip")
    if not replay_zip.is_file():
        raise RuntimeError(f"No se encontró el ZIP correspondiente a {latest_result.name}")

    safe_team_id = AUDIT_TEAM_ID.strip().upper()
    audit_dir = results_dir / "audits" / f"{result['runId']}-{safe_team_id.lower()}-{AUDIT_BASELINE}"
    audit_command = [
        str(battle_lab_python),
        "battle_lab/audit_benchmark.py",
        "--result-json", str(latest_result),
        "--replays-zip", str(replay_zip),
        "--output-dir", str(audit_dir),
        "--team-id", safe_team_id,
        "--baseline", AUDIT_BASELINE,
        "--sample-size", str(AUDIT_SAMPLE_SIZE),
    ]
    run_live(audit_command, cwd=pkmn_root, label=f"Auditar {safe_team_id} bajo el microscopio", env=battle_lab_env)

    audit_path = audit_dir / "audit.json"
    audit = json.loads(audit_path.read_text(encoding="utf-8"))
    record = audit["record"]
    paired = audit["pairedEvidence"]
    coverage = audit["replayCoverage"]
    finding_rows = "".join(
        f"<li style='margin:6px 0'><b>{item['code']}</b> ({item['confidence']}): {item['evidence']} {item['interpretation']}</li>"
        for item in audit["diagnosticFindings"]
    )
    lead_rows = "".join(
        f"<tr><td style='padding:5px 12px 5px 0'>{row['lead']}</td><td style='padding:5px 12px'>{row['games']}</td><td style='padding:5px 12px'>{row['wins']}-{row['losses']}-{row['ties']}</td><td style='padding:5px 0 5px 12px'>{row['winPercent']:.2f}%</td></tr>"
        for row in audit["observations"]["leadPerformance"][:8]
    )
    display(HTML(f"""
    <div style='padding:18px;border-radius:14px;background:#07111f;color:#e2e8f0;border:1px solid #7c3aed'>
      <div style='font-size:20px;font-weight:800;color:#c4b5fd'>🐉 Auditoría {safe_team_id} completada</div>
      <div style='margin-top:10px'>Contra <b>{AUDIT_BASELINE}</b>: <b>{record['wins']}-{record['losses']}-{record['ties']}</b> · {record['winPercent']:.2f}%</div>
      <div>Derrotas cuyo espejo ganó: <b>{paired['lossMirrorWin']}/{paired['losses']}</b> · {paired['lossMirrorWinPercent']:.2f}%</div>
      <div>Lectura: <b>{paired['interpretation']}</b></div>
      <div>Cobertura: <b>{coverage['parsed']}/{coverage['expected']}</b> replays relevantes</div>
      <ul style='margin:12px 0;padding-left:22px'>{finding_rows}</ul>
      <table style='margin-top:12px;border-collapse:collapse'><thead><tr style='color:#94a3b8'><th style='text-align:left'>Lead</th><th>Partidas</th><th>W-L-T</th><th>Win rate</th></tr></thead><tbody>{lead_rows}</tbody></table>
      <div style='margin-top:12px;color:#94a3b8'>Reporte navegable: {audit_dir / 'report.html'}</div>
      <div style='color:#94a3b8'>CSV de casos: {audit_dir / 'cases.csv'}</div>
    </div>
    """))
else:
    print("⏭️ Auditoría omitida por configuración.")
